# DeepVariant Fine-Tuning on Colab

Fine-tune the pretrained WGS checkpoint on public HG002 PE150 data, chr22.

**Prerequisites:** Upload to `Google Drive / dv_finetune/`:
```
dv_finetune/
  data/  chr22_test.bam
  ref/   GCA_000001405.15_GRCh38_no_alt_analysis_set.fa.gz
         truth_chroms_chr22.vcf
         HG002_GRCh38_1_22_v4.2.1_benchmark_noinconsistent.bed
```

**Pipeline:** Install DV (udocker) → Mount Drive → Prep ref → Download checkpoint → make_examples → Shuffle/Split → Train → Call variants → hap.py evaluation

In [ ]:
# Cell 1: GPU check + install udocker + pull DeepVariant 1.6.1
import subprocess, os, shutil, glob

r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                   capture_output=True, text=True)
print('GPU:', r.stdout.strip() if r.returncode == 0 else 'NOT FOUND')

# Clean stale udocker cache to avoid layer errors
!rm -rf ~/.udocker /content/.udocker /tmp/udocker* 2>/dev/null || true

!pip install -q udocker
!udocker --allow-root install

print('\nPulling google/deepvariant:1.6.1 (~5-15 min first time)...')
!udocker --allow-root pull google/deepvariant:1.6.1 2>&1 | tail -5

# Note: --nvidia is NOT supported in Colab's udocker (training runs on CPU)
DV_RUN = "udocker --allow-root run --volume=/content:/content google/deepvariant:1.6.1"
print(f'\nDV_RUN = {DV_RUN}')

In [ ]:
# Cell 2: Mount Google Drive + configure paths
from google.colab import drive, auth
auth.authenticate_user()
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/dv_finetune'
ROOT  = '/content/dv'

for d in ['data', 'ref', 'examples', 'shuffled', 'pretrained', 'training', 'calling', 'eval']:
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)

CFG = {
    'ref':           f'{ROOT}/ref/GCA_000001405.15_GRCh38_no_alt_analysis_set.fa',
    'ref_gz':        f'{ROOT}/ref/GCA_000001405.15_GRCh38_no_alt_analysis_set.fa.gz',
    'truth_vcf':     f'{ROOT}/ref/truth_chroms_chr22.vcf.gz',
    'conf_bed':      f'{ROOT}/ref/HG002_GRCh38_1_22_v4.2.1_benchmark_noinconsistent.bed',
    'bam':           f'{ROOT}/data/chr22_test.bam',
    'examples_pfx':  f'{ROOT}/examples/train.tfrecord.gz',
    'pretrained':    f'{ROOT}/pretrained/deepvariant.wgs.ckpt',
    'train_dir':     f'{ROOT}/training',
    'call_examples': f'{ROOT}/calling/call.tfrecord.gz',
    'call_tfr':      f'{ROOT}/calling/call_output.tfrecord.gz',
    'vcf_out':       f'{ROOT}/calling/output.vcf.gz',
    'eval_pfx':      f'{ROOT}/eval/happy',
    'region':        'chr22',
    'batch_size':    32,
    'lr':            5e-5,
    'num_steps':     200,   # increase to 10000-50000 for real training
}

print('Config ready. ROOT =', ROOT)

In [ ]:
# Cell 3: Copy data from Drive to local disk
import shutil

files = [
    (f'{DRIVE}/data/chr22_test.bam',                                         CFG['bam']),
    (f'{DRIVE}/ref/GCA_000001405.15_GRCh38_no_alt_analysis_set.fa.gz',      CFG['ref_gz']),
    (f'{DRIVE}/ref/truth_chroms_chr22.vcf',                                  f'{ROOT}/ref/truth_chroms_chr22.vcf'),
    (f'{DRIVE}/ref/HG002_GRCh38_1_22_v4.2.1_benchmark_noinconsistent.bed',  CFG['conf_bed']),
]

for src, dst in files:
    if not os.path.exists(dst):
        print(f'Copying {os.path.basename(src)}...')
        shutil.copy2(src, dst)
    print(f'  OK  {os.path.basename(dst)}  ({os.path.getsize(dst)/1e6:.0f} MB)')

In [ ]:
%%time
# Cell 4: Prep reference — decompress FASTA, index BAM, bgzip+tabix VCF
# Uses tools inside DV container (samtools, bgzip, tabix)

if not os.path.exists(CFG['ref']):
    print('Decompressing FASTA...')
    !{DV_RUN} bash -c "gunzip -k {CFG['ref_gz']}"

if not os.path.exists(CFG['ref'] + '.fai'):
    print('Indexing reference FASTA...')
    !{DV_RUN} samtools faidx {CFG['ref']}

if not os.path.exists(CFG['bam'] + '.bai'):
    print('Indexing BAM...')
    !{DV_RUN} samtools index -@ 4 {CFG['bam']}

raw_vcf = f"{ROOT}/ref/truth_chroms_chr22.vcf"
if os.path.exists(raw_vcf) and not os.path.exists(CFG['truth_vcf']):
    print('Compressing and indexing truth VCF...')
    !{DV_RUN} bash -c "bgzip -c {raw_vcf} > {CFG['truth_vcf']} && tabix -p vcf {CFG['truth_vcf']}"

for name, path in {'FASTA': CFG['ref'], 'FASTA.fai': CFG['ref']+'.fai',
                    'BAM': CFG['bam'], 'BAM.bai': CFG['bam']+'.bai',
                    'truth VCF': CFG['truth_vcf'], 'truth .tbi': CFG['truth_vcf']+'.tbi',
                    'conf BED': CFG['conf_bed']}.items():
    print(f"  [{'OK' if os.path.exists(path) else 'MISSING'}] {name}")

In [ ]:
# Cell 5: Download pretrained WGS checkpoint from GCS
ckpt_dir = f'{ROOT}/pretrained/'
if not glob.glob(f'{ckpt_dir}deepvariant.wgs.ckpt*'):
    print('Downloading WGS checkpoint from GCS (~350 MB)...')
    !gsutil -m cp -r "gs://deepvariant/models/DeepVariant/1.6.1/checkpoints/wgs/*" {ckpt_dir}
    # 如果 latest 不行，可改成 1.6.1： gs://deepvariant/models/DeepVariant/1.6.1/checkpoints/wgs/*

ckpt_files = glob.glob(f'{ckpt_dir}deepvariant.wgs.ckpt*')
print(f'{len(ckpt_files)} checkpoint files:')
for f in sorted(ckpt_files):
    print(f'  {os.path.basename(f)}  ({os.path.getsize(f)/1e6:.1f} MB)')

assert os.path.exists(CFG['pretrained'] + '.index'), 'Missing checkpoint .index file'

In [ ]:
%%time
# Cell 6: make_examples — training mode, PE150, chr22
!{DV_RUN} /opt/deepvariant/bin/make_examples \
  --mode training \
  --ref {CFG['ref']} \
  --reads {CFG['bam']} \
  --truth_variants {CFG['truth_vcf']} \
  --confident_regions {CFG['conf_bed']} \
  --examples {CFG['examples_pfx']} \
  --regions {CFG['region']} \
  --channel_list=read_base,base_quality,mapping_quality,strand,read_supports_variant,base_differs_from_ref,insert_size \
  --keep_supplementary_alignments \
  --normalize_reads \
  --task 0

recs = glob.glob(f"{ROOT}/examples/*.tfrecord.gz*")
print(f"\nGenerated {len(recs)} file(s):")
for r in recs:
    print(f"  {os.path.basename(r)}  ({os.path.getsize(r)/1e6:.1f} MB)")

In [ ]:
%%time
# Cell 7: Shuffle + split into train/tune datasets (90/10)
# DV 1.6.1 training requires both train and tune datasets
import tensorflow as tf
import random

example_files = sorted([f for f in glob.glob(f"{ROOT}/examples/train.tfrecord.gz*")
                         if not f.endswith('.json')])
assert example_files, "No tfrecord files found"

ds = tf.data.TFRecordDataset(example_files, compression_type='GZIP')
records = list(ds)
n = len(records)
print(f"Total examples: {n:,}")

random.seed(42)
random.shuffle(records)

split = int(n * 0.9)
train_records, tune_records = records[:split], records[split:]
print(f"Train: {len(train_records):,}  Tune: {len(tune_records):,}")

train_dir = f"{ROOT}/shuffled/train"
tune_dir  = f"{ROOT}/shuffled/tune"
os.makedirs(train_dir, exist_ok=True)
os.makedirs(tune_dir, exist_ok=True)

opts = tf.io.TFRecordOptions(compression_type='GZIP')
for path, recs in [(f"{train_dir}/train.tfrecord.gz", train_records),
                    (f"{tune_dir}/tune.tfrecord.gz", tune_records)]:
    writer = tf.io.TFRecordWriter(path, options=opts)
    for rec in recs:
        writer.write(rec.numpy())
    writer.close()
    print(f"Written: {path}  ({os.path.getsize(path)/1e6:.1f} MB)")

# Copy example_info.json (required by train.py next to tfrecords)
src_info = f"{ROOT}/examples/train.tfrecord.gz.example_info.json"
for d in [train_dir, tune_dir]:
    shutil.copy2(src_info, os.path.join(d, "example_info.json"))

# Write dataset config pbtxt files
train_cfg_path = f"{ROOT}/shuffled/train_dataset.pbtxt"
tune_cfg_path  = f"{ROOT}/shuffled/tune_dataset.pbtxt"

with open(train_cfg_path, 'w') as f:
    f.write(f'name: "pe150_chr22_train"\ntfrecord_path: "{train_dir}/train.tfrecord.gz"\nnum_examples: {len(train_records)}\n')
with open(tune_cfg_path, 'w') as f:
    f.write(f'name: "pe150_chr22_tune"\ntfrecord_path: "{tune_dir}/tune.tfrecord.gz"\nnum_examples: {len(tune_records)}\n')

print(f"\nDataset configs: {train_cfg_path}, {tune_cfg_path}")
CFG['train_dataset_pbtxt'] = train_cfg_path
CFG['tune_dataset_pbtxt']  = tune_cfg_path

In [ ]:
%%time
# Cell 8: Fine-tune — DV 1.6.1 Keras training API
# Uses ml_collections config file + /opt/deepvariant/bin/train

config_py = f"{ROOT}/training/config.py"
with open(config_py, 'w') as f:
    f.write(f"""\
import ml_collections

def get_config():
    config = ml_collections.ConfigDict()

    # Dataset paths
    config.train_dataset_pbtxt = '{CFG['train_dataset_pbtxt']}'
    config.tune_dataset_pbtxt = '{CFG['tune_dataset_pbtxt']}'

    # Model
    config.model_type = 'inception_v3'
    config.init_checkpoint = '{CFG['pretrained']}'
    config.init_backbone_with_imagenet = False
    config.weight_decay = 0.00004
    config.backbone_dropout_rate = 0.2

    # Training hyperparams
    config.batch_size = {CFG['batch_size']}
    config.num_epochs = 1
    config.learning_rate = {CFG['lr']}
    config.learning_rate_num_epochs_per_decay = 2.0
    config.learning_rate_decay_rate = 0.94
    config.warmup_steps = 0
    config.label_smoothing = 0.0001

    # Optimizer
    config.optimizer = 'rmsprop'
    config.rho = 0.9
    config.momentum = 0.9
    config.epsilon = 1.0

    # Data pipeline
    config.prefetch_buffer_bytes = 16 * 1024 * 1024
    config.input_read_threads = 4
    config.shuffle_buffer_elements = 1000

    # Logging / checkpointing
    config.log_every_steps = 10
    config.tune_every_steps = 50
    config.best_checkpoint_metric = 'tune/f1_weighted'
    config.early_stopping_patience = 0
    config.num_validation_examples = 0

    return config
""")
print(f"Config written: {config_py}")

!{DV_RUN} /opt/deepvariant/bin/train \
  --config="{config_py}" \
  --experiment_dir="{CFG['train_dir']}" \
  --limit={CFG['num_steps']}

ckpts = sorted(glob.glob(f"{CFG['train_dir']}/checkpoints/*"))
print(f"\nCheckpoint files: {len(ckpts)}")
for c in ckpts:
    print(f"  {os.path.basename(c)}")

In [ ]:
%%time
# Cell 9: Variant calling with fine-tuned checkpoint

# Find latest checkpoint (DV 1.6.1 Keras format: ckpt-<step>)
ckpt_files = sorted(glob.glob(f"{CFG['train_dir']}/checkpoints/ckpt-*.index"))
CKPT = ckpt_files[-1].replace('.index', '') if ckpt_files else CFG['pretrained']
print(f'Using checkpoint: {CKPT}')

# make_examples — calling mode
!{DV_RUN} /opt/deepvariant/bin/make_examples \
  --mode calling \
  --ref {CFG['ref']} \
  --reads {CFG['bam']} \
  --examples {CFG['call_examples']} \
  --regions {CFG['region']} \
  --channel_list=read_base,base_quality,mapping_quality,strand,read_supports_variant,base_differs_from_ref,insert_size \
  --keep_supplementary_alignments \
  --normalize_reads \
  --task 0

call_tfrs = [f for f in glob.glob(f"{CFG['call_examples']}*") if not f.endswith('.json')]
print(f"Call examples: {len(call_tfrs)} file(s)")
assert call_tfrs, "make_examples (calling) produced no output"

# call_variants
!{DV_RUN} /opt/deepvariant/bin/call_variants \
  --outfile {CFG['call_tfr']} \
  --examples {CFG['call_examples']} \
  --checkpoint {CKPT}

# postprocess_variants
!{DV_RUN} /opt/deepvariant/bin/postprocess_variants \
  --ref {CFG['ref']} \
  --infile {CFG['call_tfr']} \
  --outfile {CFG['vcf_out']}

print('\nDone. Output VCF:', CFG['vcf_out'])

In [ ]:
%%time
# Cell 10: Evaluate with hap.py

print('Pulling hap.py image...')
!udocker --allow-root pull pkrusche/hap.py 2>&1 | tail -5
HAPPY_RUN = "udocker --allow-root run --volume=/content:/content pkrusche/hap.py"

!{HAPPY_RUN} /opt/hap.py/bin/hap.py \
  {CFG['truth_vcf']} \
  {CFG['vcf_out']} \
  -f {CFG['conf_bed']} \
  -r {CFG['ref']} \
  -o {CFG['eval_pfx']} \
  -l {CFG['region']} \
  --engine=vcfeval --threads=4

import pandas as pd
csv_file = f"{CFG['eval_pfx']}.summary.csv"
if os.path.exists(csv_file):
    df = pd.read_csv(csv_file)
    display(df[['Type', 'Filter', 'METRIC.Recall', 'METRIC.Precision', 'METRIC.F1_Score']])
    df.to_csv(f'{DRIVE}/happy_results.csv', index=False)
    print(f'Saved to {DRIVE}/happy_results.csv')
else:
    print('hap.py output not found — check errors above')